<a href="https://colab.research.google.com/github/Gursharan-Reddy/InterSpark-Data-Science-Internship/blob/main/Task2_Loan_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhanupratapbiswas/loan-approval-prediction-case-study")

print("Path to dataset files:", path)

100%|██████████| 9.72k/9.72k [00:00<00:00, 11.5MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/bhanupratapbiswas/loan-approval-prediction-case-study/versions/1


In [2]:
# 1. IMPORT LIBRARIES AND DATASET
import kagglehub
import pandas as pd
import numpy as np
import os
import warnings

# ML and Preprocessing Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

warnings.filterwarnings('ignore')

# Download and load the dataset
print("Downloading dataset...")
path = kagglehub.dataset_download("bhanupratapbiswas/loan-approval-prediction-case-study")
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]

if csv_files:
    csv_path = os.path.join(path, csv_files[0])
    df = pd.read_csv(csv_path)
    print("Dataset loaded successfully!\n")
else:
    print("Error: No CSV file found.")

# ---------------------------------------------------------
# 2. DATA PREPROCESSING
print("--- Basic Info Before Cleaning ---")
print(f"Dataset Shape: {df.shape}")

# Drop 'Loan_ID' as it is just an identifier and useless for prediction
if 'Loan_ID' in df.columns:
    df = df.drop('Loan_ID', axis=1)

# Handle Missing Values safely
# Fill categorical columns with the mode (most frequent) and numerical with median
for col in df.columns:
    if df[col].dtype == 'object':
        df[col].fillna(df[col].mode()[0], inplace=True)
    else:
        df[col].fillna(df[col].median(), inplace=True)

# Identify the target column (usually 'Loan_Status')
target_col = 'Loan_Status'
if target_col not in df.columns:
    target_col = df.columns[-1] # Fallback to the last column if named differently

# Encode Categorical Variables
le = LabelEncoder()
for col in df.select_dtypes(include=['object']).columns:
    df[col] = le.fit_transform(df[col])

# Define Features (X) and Target (y)
X = df.drop(target_col, axis=1)
y = df[target_col]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling (Important for algorithms like Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------------------------------------------------
# 3. HANDLE CLASS IMBALANCE
print("\nApplying SMOTE to handle class imbalance...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"Original training target distribution:\n{y_train.value_counts()}")
print(f"Resampled training target distribution:\n{y_train_resampled.value_counts()}\n")

# ---------------------------------------------------------
# 4. MODEL TRAINING & EVALUATION

# Model 1: Logistic Regression
print("--- Logistic Regression Model Evaluation ---")
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_resampled, y_train_resampled)

y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred_lr))
print(f"Logistic Regression ROC-AUC Score: {roc_auc_score(y_test, y_prob_lr):.4f}\n")

# Model 2: Random Forest
print("--- Random Forest Model Evaluation ---")
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X_train_resampled, y_train_resampled)

y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred_rf))
print(f"Random Forest ROC-AUC Score: {roc_auc_score(y_test, y_prob_rf):.4f}")

Using Colab cache for faster access to the 'loan-approval-prediction-case-study' dataset.
Dataset loaded successfully!

--- Basic Info Before Cleaning ---
Dataset Shape: (614, 13)

Applying SMOTE to handle class imbalance...
Original training target distribution:
Loan_Status
1    342
0    149
Name: count, dtype: int64
Resampled training target distribution:
Loan_Status
0    342
1    342
Name: count, dtype: int64

--- Logistic Regression Model Evaluation ---
              precision    recall  f1-score   support

           0       0.76      0.51      0.61        43
           1       0.78      0.91      0.84        80

    accuracy                           0.77       123
   macro avg       0.77      0.71      0.73       123
weighted avg       0.77      0.77      0.76       123

Logistic Regression ROC-AUC Score: 0.7340

--- Random Forest Model Evaluation ---
              precision    recall  f1-score   support

           0       0.63      0.44      0.52        43
           1       0